# 02 · MSI / MMR Sensitivity Analysis
### Revision analysis for DCR-D-25-01148 — responds to **Reviewer #1, Comment 3**

> *"You didn't incorporate MMR status, which is now widely available… How do you think that impacted your model?"*

**本 notebook 做什麼.** (1) 量化 MSI/MMR 與 RAS/BRAF 在兩 cohort 的**實際可得率**；(2) 以作者原始 harness（seed 8251）建立 **四變數 + MSI** 之 sensitivity model，檢視加入 MSI 是否改善 discrimination / rule-out；(3) 從而說明為何最終精簡模型未納入 MSI。


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from xgboost import XGBClassifier

def find_data_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base/'local_data', base/'stage_III_colon_edr'/'local_data'):
            if cand.exists(): return cand
    raise FileNotFoundError('local_data not found')
DATA=find_data_dir(); SEED=8251
deriv=pd.read_parquet(DATA/'all_cases_prepared_for_ML.parquet')
ext=pd.read_csv(DATA/'data_typed_ext.csv')
print('Derivation n=',len(deriv),' External n=',len(ext))

Derivation n= 331  External n= 142


## 1. 分子檢測可得率（實證「非普遍可得」）

Reviewer 認為 MMR 現已普遍可得。下面用**本研究兩 cohort 的實際數字**呈現跨院落差，這正是支持「精簡、免分子模型」的實證。


In [2]:
def availability(df,name):
    n=len(df); msi=df['MSI_High']
    print(f'{name:11s} (n={n}): MSI 可得 {msi.notna().sum()}/{n} ({msi.notna().mean()*100:.0f}%), '
          f'MSI-high={int((msi==1).sum())}, 缺失={int(msi.isna().sum())}')
availability(deriv,'DERIVATION'); availability(ext,'EXTERNAL')
print('\\n→ 衍生組近乎完整(99%)、外部僅約 57%。跨院可得率落差本身即為 reviewer 問題的關鍵回答：')
print('  MMR/MSI 在真實世界（尤其非醫學中心）並非一致可得，這正是本研究採免分子精簡模型的動機。')

# RAS/BRAF 稀疏度（衍生組原始病理報告）
try:
    xl=pd.ExcelFile(DATA/'Patient_Cohort.xlsx'); pa=xl.parse('Pathology')
    for c in ['MSI status','RAS mutation','BRAF mutation']:
        if c in pa.columns: print(f'  [原始病理報告] {c:16s}: 非空 {int(pa[c].notna().sum())} / {len(pa)}')
except Exception as e:
    print('  (Pathology 原始欄位讀取略過:', e, ')')

DERIVATION  (n=331): MSI 可得 329/331 (99%), MSI-high=26, 缺失=2
EXTERNAL    (n=142): MSI 可得 81/142 (57%), MSI-high=6, 缺失=61
\n→ 衍生組近乎完整(99%)、外部僅約 57%。跨院可得率落差本身即為 reviewer 問題的關鍵回答：
  MMR/MSI 在真實世界（尤其非醫學中心）並非一致可得，這正是本研究採免分子精簡模型的動機。


  [原始病理報告] MSI status      : 非空 505 / 2508
  [原始病理報告] RAS mutation    : 非空 124 / 2508
  [原始病理報告] BRAF mutation   : 非空 16 / 2508


## 2. 四變數 + MSI 之 sensitivity model（衍生組）

以作者原始 harness（seed 8251, KNNImputer(k=5) → isotonic-calibrated XGBoost, 5 outer folds, Table S2 超參數）比較：
- **Base**：四變數（AJCC one-hot, PNI, LNR, Differentiation）— 重現 AUC≈0.698
- **+MSI**：四變數 + `MSI_High`（衍生組 99% 可得）

AUC 以 mean-across-folds 報告（與 manuscript 一致）。


In [3]:
XGB=dict(n_estimators=50,max_depth=2,learning_rate=0.05,gamma=1.0,min_child_weight=1,subsample=0.9,
         colsample_bytree=0.6,reg_alpha=0.5,reg_lambda=1.0,eval_metric='logloss',random_state=SEED,n_jobs=1)
y=deriv['edr_18m'].astype(int); ratio=float((y==0).sum()/(y==1).sum())

def base_X(df):
    X=pd.get_dummies(df[['AJCC_Substage','PNI','LNR','Differentiation']],columns=['AJCC_Substage'])
    X['PNI']=X['PNI'].astype(float); X['Differentiation']=X['Differentiation'].astype(float)
    return X.replace([np.inf,-np.inf],np.nan)

def run_harness(X,y):
    outer=StratifiedKFold(5,shuffle=True,random_state=SEED); aucs=[]; oof=np.zeros(len(y))
    for tr,te in outer.split(X,y):
        m=CalibratedClassifierCV(XGBClassifier(**XGB,scale_pos_weight=ratio),method='isotonic',cv=3)
        pipe=Pipeline([('imp',KNNImputer(n_neighbors=5)),('m',m)]); pipe.fit(X.iloc[tr],y.iloc[tr])
        p=pipe.predict_proba(X.iloc[te])[:,1]; aucs.append(roc_auc_score(y.iloc[te],p)); oof[te]=p
    return float(np.mean(aucs)), oof

Xbase=base_X(deriv)
Xmsi=Xbase.copy(); Xmsi['MSI_High']=deriv['MSI_High'].astype(float)
auc_base,oof_base=run_harness(Xbase,y)
auc_msi, oof_msi =run_harness(Xmsi, y)
print(f'Base (4-var)   mean-fold AUC = {auc_base:.3f}')
print(f'4-var + MSI    mean-fold AUC = {auc_msi:.3f}')
print(f'Delta AUC (加入 MSI) = {auc_msi-auc_base:+.3f}')

Base (4-var)   mean-fold AUC = 0.698
4-var + MSI    mean-fold AUC = 0.705
Delta AUC (加入 MSI) = +0.007


In [4]:
# rule-out NPV/sensitivity at fixed 0.120 cutoff (原研究 OOF-derived)
def ro(y,p,cut=0.120):
    yh=(np.asarray(p)>=cut).astype(int); tn,fp,fn,tp=confusion_matrix(y,yh).ravel()
    return tp/(tp+fn), tn/(tn+fp), (tn/(tn+fn) if (tn+fn) else np.nan)
for name,oof in [('Base (4-var)',oof_base),('4-var + MSI',oof_msi)]:
    se,sp,npv=ro(y.values,oof)
    print(f'{name:14s}: sensitivity={se*100:.1f}%  specificity={sp*100:.1f}%  NPV={npv*100:.1f}%')

Base (4-var)  : sensitivity=77.4%  specificity=45.7%  NPV=89.8%
4-var + MSI   : sensitivity=80.6%  specificity=52.0%  NPV=92.1%


## 3. 小結（供 Response letter / Supplementary Table）

- **可得率落差**（衍生 99% vs 外部 57%）本身即回答 reviewer：MMR/MSI 在跨機構真實世界並非一致可得。
- **加入 MSI 對 discrimination 的影響**見上 ΔAUC；若無實質提升，正支持「精簡模型已足夠、且免除對分子檢測的依賴」之論點（與全文定位一致）。
- 外部 cohort MSI 僅 57% 可得，故 sensitivity model 以**衍生組（99% 可得、well-powered）**為主；此點於 Supplementary 註明。

> 本節結果彙整為新 **Supplementary Table（MSI sensitivity）**。
